<a href="https://colab.research.google.com/github/pranuk050-pixel/Pyspark_Programming/blob/main/10_09_26_q3_complexdata_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('window_functions').getOrCreate()

In [ ]:

data = [(1, ['sales', 'finance', 'HR']), (2, ['sales', 'HR']), (3, ['sales', 'security'])]
df = spark.createDataFrame(data, ['id', 'dept'])
df.show()
df.printSchema()

# print for howmany depts each emp is working
# to process array column, explode()
df1= df.withColumn('dept', explode('dept'))
df1.show()
df1.printSchema()

# to generate array column ==> collect_list()
df2= df1.groupBy("id").agg(collect_list('dept').alias('dept'))
df2.show()
df2.printSchema()

+---+--------------------+
| id|                dept|
+---+--------------------+
|  1|[sales, finance, HR]|
|  2|         [sales, HR]|
|  3|   [sales, security]|
+---+--------------------+

root
 |-- id: long (nullable = true)
 |-- dept: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---+--------+
| id|    dept|
+---+--------+
|  1|   sales|
|  1| finance|
|  1|      HR|
|  2|   sales|
|  2|      HR|
|  3|   sales|
|  3|security|
+---+--------+

root
 |-- id: long (nullable = true)
 |-- dept: string (nullable = true)

+---+--------------------+
| id|                dept|
+---+--------------------+
|  1|[sales, finance, HR]|
|  3|   [sales, security]|
|  2|         [sales, HR]|
+---+--------------------+

root
 |-- id: long (nullable = true)
 |-- dept: array (nullable = false)
 |    |-- element: string (containsNull = false)



In [ ]:

json_df = spark.read.json('details.json')
json_df.show()

+---------+------+
|     city|  name|
+---------+------+
|Bangalore|charan|
+---------+------+



In [ ]:
df1= spark.read.json('devices.json')
df1.show()

+---------+--------------------+--------+---+----+------+----+-------------------+-------+
|device_id|         device_name|humidity|lat|long| scale|temp|          timestamp|zipcode|
+---------+--------------------+--------+---+----+------+----+-------------------+-------+
|        1|sensor-mac$$$ %-a...|      80| 81|  57|Celius|  30|1.447975123509765E9|  95353|
|        2|sensor-mac-able9b...|      36| 13|  56|Celius|  14|1.447975124005187E9|  96484|
|        3|sensor-mac-aboutR...|      30| 35|  54|Celius|  17|1.447975124054221E9|  96402|
|        4|sensor-mac-across...|      35| 36|  40|Celius|   6|1.447975124102157E9|  96025|
|        5|sensor-mac-afterE...|      62| 90|  91|Celius|  23|1.447975124150419E9|  95638|
|        6|sensor-mac-allBMO...|      97| 76|  77|Celius|   8|1.447975124197694E9|  95478|
|        7|sensor-mac-almost...|      30| 95|  76|Celius|   5|1.447975124246103E9|  95499|
|        8|sensor-mac-alsosT...|      74| 95|  79|Celius|  25|1.447975124291892E9|  94857|

In [ ]:
donut_df= spark.read.json('donut.json', multiLine= True)

donut_df.printSchema()
donut_df.show()
donut_df.withColumn('image_height', col('image.height'))\
        .withColumn('image_url', col('image.url')) \
        .withColumn('image_width', col('image.width')) \
        .withColumn('thumbnail_height', col('thumbnail.height'))\
        .withColumn('thumbnail_url', col('thumbnail.url')) \
        .withColumn('thumbnail_width', col('thumbnail.width')) \
        .drop('image', 'thumbnail').show()

root
 |-- id: string (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- height: long (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- width: long (nullable = true)
 |-- name: string (nullable = true)
 |-- thumbnail: struct (nullable = true)
 |    |-- height: long (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- width: long (nullable = true)
 |-- type: string (nullable = true)

+----+--------------------+----+--------------------+-----+
|  id|               image|name|           thumbnail| type|
+----+--------------------+----+--------------------+-----+
|0001|{200, images/0001...|Cake|{32, images/thumb...|donut|
+----+--------------------+----+--------------------+-----+

+----+----+-----+------------+---------------+-----------+----------------+--------------------+---------------+
|  id|name| type|image_height|      image_url|image_width|thumbnail_height|       thumbnail_url|thumbnail_width|
+----+----+-----+------------+-----------

In [ ]:
donut_df.selectExpr('id', 'image.*', 'name', 'thumbnail.*','type').show()

+----+------+---------------+-----+----+------+--------------------+-----+-----+
|  id|height|            url|width|name|height|                 url|width| type|
+----+------+---------------+-----+----+------+--------------------+-----+-----+
|0001|   200|images/0001.jpg|  200|Cake|    32|images/thumbnails...|   32|donut|
+----+------+---------------+-----+----+------+--------------------+-----+-----+



In [ ]:
from pyspark.sql.types import StructType
# when you process struct column, its called as flattening the columns

def flatten_df(df):
    def get_columns(schema, prefix=""):
        columns = []

        for field in schema.fields:
            column_name = (
                f"{prefix}.{field.name}"
                if prefix
                else field.name
            )

            if type(field.dataType) == StructType:
                columns.extend(
                    get_columns(
                        field.dataType,
                        column_name
                    )
                )
            else:
                alias_name = column_name.replace(".", "_")

                columns.append(
                    df[column_name].alias(alias_name)
                )

        return columns

    return df.select(*get_columns(df.schema))

final_df = flatten_df(donut_df)
final_df.show()
final_df.printSchema()

+----+------------+---------------+-----------+----+----------------+--------------------+---------------+-----+
|  id|image_height|      image_url|image_width|name|thumbnail_height|       thumbnail_url|thumbnail_width| type|
+----+------------+---------------+-----------+----+----------------+--------------------+---------------+-----+
|0001|         200|images/0001.jpg|        200|Cake|              32|images/thumbnails...|             32|donut|
+----+------------+---------------+-----------+----+----------------+--------------------+---------------+-----+

root
 |-- id: string (nullable = true)
 |-- image_height: long (nullable = true)
 |-- image_url: string (nullable = true)
 |-- image_width: long (nullable = true)
 |-- name: string (nullable = true)
 |-- thumbnail_height: long (nullable = true)
 |-- thumbnail_url: string (nullable = true)
 |-- thumbnail_width: long (nullable = true)
 |-- type: string (nullable = true)



In [ ]:
results_df = spark.read.json('results.json', multiLine=True)
results_df.printSchema()

root
 |-- nationality: string (nullable = true)
 |-- results: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- user: struct (nullable = true)
 |    |    |    |-- INSEE: string (nullable = true)
 |    |    |    |-- cell: string (nullable = true)
 |    |    |    |-- dob: long (nullable = true)
 |    |    |    |-- email: string (nullable = true)
 |    |    |    |-- gender: string (nullable = true)
 |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |-- city: string (nullable = true)
 |    |    |    |    |-- state: string (nullable = true)
 |    |    |    |    |-- street: string (nullable = true)
 |    |    |    |    |-- zip: long (nullable = true)
 |    |    |    |-- md5: string (nullable = true)
 |    |    |    |-- name: struct (nullable = true)
 |    |    |    |    |-- first: string (nullable = true)
 |    |    |    |    |-- last: string (nullable = true)
 |    |    |    |    |-- title: string (nullable = true)
 |    |    |  

In [ ]:
df1= results_df.withColumn('results', explode(col('results')))
df1.printSchema()
final_df = flatten_df(df1)
final_df.show()
final_df.printSchema()

root
 |-- nationality: string (nullable = true)
 |-- results: struct (nullable = true)
 |    |-- user: struct (nullable = true)
 |    |    |-- INSEE: string (nullable = true)
 |    |    |-- cell: string (nullable = true)
 |    |    |-- dob: long (nullable = true)
 |    |    |-- email: string (nullable = true)
 |    |    |-- gender: string (nullable = true)
 |    |    |-- location: struct (nullable = true)
 |    |    |    |-- city: string (nullable = true)
 |    |    |    |-- state: string (nullable = true)
 |    |    |    |-- street: string (nullable = true)
 |    |    |    |-- zip: long (nullable = true)
 |    |    |-- md5: string (nullable = true)
 |    |    |-- name: struct (nullable = true)
 |    |    |    |-- first: string (nullable = true)
 |    |    |    |-- last: string (nullable = true)
 |    |    |    |-- title: string (nullable = true)
 |    |    |-- password: string (nullable = true)
 |    |    |-- phone: string (nullable = true)
 |    |    |-- picture: struct (nullable = t

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

data =[('IN',), ('USA',), ('UK',), ('IR',)]
country_codes = spark.createDataFrame(data, ['code'])
country_codes.show()

country_names = {'IN': 'India', 'USA' :'United States of America', 'UK': 'United Kingdom', 'IR': 'Ireland'}

@udf(returnType= StringType())
def get_country_name(code):
  return country_names.get(code)

names_df = country_codes.withColumn('country_name', get_country_name(col('code')))
names_df.show()

+----+
|code|
+----+
|  IN|
| USA|
|  UK|
|  IR|
+----+

+----+--------------------+
|code|        country_name|
+----+--------------------+
|  IN|               India|
| USA|United States of ...|
|  UK|      United Kingdom|
|  IR|             Ireland|
+----+--------------------+



In [ ]:
data =[('charan', 'c' , 'Mr.'), ('Rahul', 'R' , 'Mr.'), ('Arun', 'R' , 'Mr.')]
df = spark.createDataFrame(data, ['f_name', 'l_name', 'title'])

df.show()

df1 =df.withColumn('full_name',struct('f_name', 'l_name', 'title'))
df1.show()
df1.printSchema()

+------+------+-----+
|f_name|l_name|title|
+------+------+-----+
|charan|     c|  Mr.|
| Rahul|     R|  Mr.|
|  Arun|     R|  Mr.|
+------+------+-----+

+------+------+-----+----------------+
|f_name|l_name|title|       full_name|
+------+------+-----+----------------+
|charan|     c|  Mr.|{charan, c, Mr.}|
| Rahul|     R|  Mr.| {Rahul, R, Mr.}|
|  Arun|     R|  Mr.|  {Arun, R, Mr.}|
+------+------+-----+----------------+

root
 |-- f_name: string (nullable = true)
 |-- l_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- full_name: struct (nullable = false)
 |    |-- f_name: string (nullable = true)
 |    |-- l_name: string (nullable = true)
 |    |-- title: string (nullable = true)



In [ ]:
from textwrap import indent
import json

url = 'https://randomuser.me/api/0.8/?results=1'

import requests

response = requests.get(url)
json_data = response.json()
print(type(json_data))

with open('url_data.json', 'w') as f:
  json.dump(json_data, f, indent=4)

df = spark.read.json('url_data.json', multiLine=True)
df.printSchema()

<class 'dict'>
root
 |-- nationality: string (nullable = true)
 |-- results: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- user: struct (nullable = true)
 |    |    |    |-- cell: string (nullable = true)
 |    |    |    |-- dob: long (nullable = true)
 |    |    |    |-- email: string (nullable = true)
 |    |    |    |-- gender: string (nullable = true)
 |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |-- city: string (nullable = true)
 |    |    |    |    |-- state: string (nullable = true)
 |    |    |    |    |-- street: string (nullable = true)
 |    |    |    |    |-- zip: long (nullable = true)
 |    |    |    |-- md5: string (nullable = true)
 |    |    |    |-- name: struct (nullable = true)
 |    |    |    |    |-- first: string (nullable = true)
 |    |    |    |    |-- last: string (nullable = true)
 |    |    |    |    |-- title: string (nullable = true)
 |    |    |    |-- password: string (nullable = tr